In [1]:
import sys, torch
print(sys.version)        
print(torch.__version__)  

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
2.9.0+cu126


In [2]:
!pip install torch torchvision --quiet
!pip install torch-geometric --quiet
!pip install matplotlib seaborn scipy tqdm --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.1 MB/s eta 0:00:00


In [ ]:
import glob, sys, torch

# Find actual wheel locations
wheels = glob.glob('/kaggle/input/**/*.whl', recursive=True)
for w in wheels:
    print(w)

print(f"\nPython: {sys.version}")
print(f"Torch:  {torch.__version__}")

# Pytorch Cuda Wheels to access PyG scatter sparse 

In [ ]:
import subprocess, sys, os, shutil

wheels = [
    '/kaggle/input/datasets/aaryaupi/pyotorch-kaggle-latest-wheels/torch_scatter-2.1.2pt29cpu-cp312-cp312-linux_x86_64.whl',
    '/kaggle/input/datasets/aaryaupi/pyotorch-kaggle-latest-wheels/torch_sparse-0.6.18pt29cpu-cp312-cp312-linux_x86_64.whl',
    '/kaggle/input/datasets/aaryaupi/pyotorch-kaggle-latest-wheels/torch_cluster-1.6.3pt29cpu-cp312-cp312-linux_x86_64.whl',
    '/kaggle/input/datasets/aaryaupi/pyotorch-kaggle-latest-wheels/torch_spline_conv-1.2.2pt29cpu-cp312-cp312-linux_x86_64.whl',
]

# Clean names pip can parse without confusion
clean_names = [
    'torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl',
    'torch_sparse-0.6.18-cp312-cp312-linux_x86_64.whl',
    'torch_cluster-1.6.3-cp312-cp312-linux_x86_64.whl',
    'torch_spline_conv-1.2.2-cp312-cp312-linux_x86_64.whl',
]

WORK = '/kaggle/working/wheels'
os.makedirs(WORK, exist_ok=True)

# Copy and rename
for src, clean in zip(wheels, clean_names):
    dst = os.path.join(WORK, clean)
    shutil.copy2(src, dst)
    print(f"Copied → {clean}")

# Now install from clean names
for clean in clean_names:
    path = os.path.join(WORK, clean)
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', path, '--no-deps', '-q'],
        capture_output=True, text=True
    )
    print(f"{'' if r.returncode==0 else ''} {clean[:35]} {r.stderr[:100] if r.returncode!=0 else ''}")

# Verify
print()
for mod in ['torch_scatter', 'torch_sparse', 'torch_cluster', 'torch_spline_conv']:
    try:
        __import__(mod)
        print(f"fOUND{mod}")
    except ModuleNotFoundError:
        print(f"Not found {mod}")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
from collections import Counter, defaultdict
from tqdm import tqdm
import scipy.spatial as spatial

import torch
import torch_geometric
from torch_geometric.datasets import ModelNet
from torch_geometric.transforms import SamplePoints, NormalizeScale, KNNGraph
import torch_geometric.transforms as T
from torch_geometric.data import DataLoader
from torch_cluster import knn_graph

In [ ]:
import os, torch, numpy as np, random
from torch_geometric.data import Data
from tqdm.auto import tqdm

RAW_ROOT   = '/kaggle/input/datasets/balraj98/modelnet40-princeton-3d-object-dataset/ModelNet40'
NUM_POINTS = 1024

CLASSES      = sorted(os.listdir(RAW_ROOT))
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

def read_off(filepath):
    with open(filepath, 'r') as f:
        header = f.readline().strip()
        if 'OFF' not in header:
            return None
        # Handle "OFF1234 567 0" vs "OFF\n1234 567 0"
        if len(header) > 3:
            counts = header[3:].strip().split()
        else:
            counts = f.readline().strip().split()
        n_verts = int(counts[0])
        verts = []
        for _ in range(n_verts):
            line = f.readline().strip().split()
            if len(line) >= 3:
                verts.append([float(x) for x in line[:3]])
    return np.array(verts, dtype=np.float32) if verts else None

def process_pointcloud(verts, n_points=1024):
    """
    Sample n_points from vertices + estimate normals via PCA on local neighborhood.
    Returns: pos [N,3], norm [N,3]
    """
    # ── Sample points ──────────────────────────────────────────────────
    if len(verts) >= n_points:
        idx = np.random.choice(len(verts), n_points, replace=False)
    else:
        idx = np.random.choice(len(verts), n_points, replace=True)
    pts = verts[idx]

    pts = pts - pts.mean(axis=0)
    pts = pts / (np.linalg.norm(pts, axis=1).max() + 1e-10)

    from scipy.spatial import cKDTree
    tree  = cKDTree(pts)
    _, nn_idx = tree.query(pts, k=10)   # [N, 10]

    normals = np.zeros_like(pts)
    for i in range(len(pts)):
        neighbors = pts[nn_idx[i]]                      # [10, 3]
        centered  = neighbors - neighbors.mean(axis=0)  # center
        cov       = centered.T @ centered               # [3, 3]
        eigvals, eigvecs = np.linalg.eigh(cov)
        normals[i] = eigvecs[:, 0]                      # smallest eigenvector = normal

    # Normalize normals to unit length
    norms_len = np.linalg.norm(normals, axis=1, keepdims=True)
    normals   = normals / (norms_len + 1e-10)

    return pts.astype(np.float32), normals.astype(np.float32)


def load_split(split='train'):
    """
    Load all .off files for a split.
    Returns list of PyG Data objects with:
        data.pos  : [1024, 3]  xyz coordinates
        data.norm : [1024, 3]  surface normals  
        data.x    : [1024, 6]  6D node features = concat(pos, norm)
        data.y    : [1]        class label
    """
    data_list = []
    all_files = []

    for cls in CLASSES:
        folder = os.path.join(RAW_ROOT, cls, split)
        if not os.path.exists(folder):
            continue
        for fname in os.listdir(folder):
            if fname.endswith('.off'):
                all_files.append((os.path.join(folder, fname), class_to_idx[cls]))

    for fpath, label in tqdm(all_files, desc=f'Loading {split}'):
        try:
            verts = read_off(fpath)
            if verts is None or len(verts) < 10:
                continue
            pts, nrm = process_pointcloud(verts, NUM_POINTS)

            data = Data(
                pos  = torch.tensor(pts),            # [1024, 3]
                norm = torch.tensor(nrm),            # [1024, 3]
                x    = torch.cat([                   # [1024, 6]  ← 6D node features
                           torch.tensor(pts),
                           torch.tensor(nrm)
                       ], dim=1),
                y    = torch.tensor([label], dtype=torch.long)
            )
            data_list.append(data)
        except:
            pass

    return data_list

print("Loading train split...")
train_list = load_split('train')
print("Loading test split...")
test_list  = load_split('test')

print(f"\nTrain samples : {len(train_list)}")
print(f"Test  samples : {len(test_list)}")

s = train_list[0]
print(f"\npos shape     : {s.pos.shape}   ← xyz")
print(f"norm shape    : {s.norm.shape}  ← normals")
print(f"x shape       : {s.x.shape}    ← 6D node features")
print(f"label         : {s.y.item()} = '{CLASSES[s.y.item()]}'")

torch.save({
    'train'  : train_list,
    'test'   : test_list,
    'classes': CLASSES
}, '/kaggle/working/modelnet40_cache.pt')

size_mb = os.path.getsize('/kaggle/working/modelnet40_cache.pt') / 1e6
print(f"{size_mb:.1f} MB")

In [ ]:
from collections import Counter
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# ── Replace train_dataset.categories with our CLASSES list ──────────
train_counts = Counter()
test_counts  = Counter()

for data in tqdm(train_list, desc='Counting train'):
    train_counts[CLASSES[data.y.item()]] += 1

for data in tqdm(test_list, desc='Counting test'):
    test_counts[CLASSES[data.y.item()]] += 1

# Sort by train count descending
sorted_classes = sorted(train_counts.keys(), key=lambda c: train_counts[c], reverse=True)
train_vals     = [train_counts[c] for c in sorted_classes]
test_vals      = [test_counts[c]  for c in sorted_classes]

# Imbalance ratio
imbalance_ratio = max(train_vals) / min(train_vals)
print(f"Imbalance ratio (max/min train samples): {imbalance_ratio:.1f}x")
print(f"Max class : {sorted_classes[0]}  ({train_vals[0]} samples)")
print(f"Min class : {sorted_classes[-1]} ({train_vals[-1]} samples)")

# ── Plot ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(22, 7))
x     = np.arange(len(sorted_classes))
width = 0.4

axes[0].bar(x - width/2, train_vals, width, label='Train', color='steelblue', alpha=0.85)
axes[0].bar(x + width/2, test_vals,  width, label='Test',  color='tomato',    alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(sorted_classes, rotation=90, fontsize=8)
axes[0].set_ylabel('Sample Count')
axes[0].set_title('ModelNet40 — Class Distribution (Train vs Test)')
axes[0].axhline(np.mean(train_vals), color='navy', linestyle='--', alpha=0.5, label='Mean train')
axes[0].legend()

# Train/test ratio per class
ratios = [train_counts[c] / (test_counts.get(c, 0) + 1e-6) for c in sorted_classes]
axes[1].bar(x, ratios, color='mediumseagreen', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(sorted_classes, rotation=90, fontsize=8)
axes[1].axhline(4.0, color='red', linestyle='--', alpha=0.7, label='Expected 4:1 ratio')
axes[1].set_ylabel('Train / Test Ratio')
axes[1].set_title('Train-to-Test Ratio Per Class (Should be ~4)')
axes[1].legend()

plt.tight_layout()
plt.savefig('artifact1_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: artifact1_class_distribution.png")

In [ ]:
from collections import defaultdict

def get_samples_per_class(data_list, n_per_class=3):
    class_samples = defaultdict(list)
    for data in data_list:
        cls = CLASSES[data.y.item()]          # ← fixed
        if len(class_samples[cls]) < n_per_class:
            class_samples[cls].append(data)
        if all(len(v) >= n_per_class for v in class_samples.values()):
            break
    return class_samples

class_samples = get_samples_per_class(train_list, n_per_class=2)
CLASSES_TO_PLOT = ['chair', 'airplane', 'lamp', 'table', 'plant', 'cone',
                   'guitar', 'car', 'laptop', 'flower_pot']

fig = plt.figure(figsize=(20, 8))
for idx, cls_name in enumerate(CLASSES_TO_PLOT):
    if cls_name not in class_samples:
        continue
    data = class_samples[cls_name][0]
    pts  = data.pos.numpy()

    ax = fig.add_subplot(2, 5, idx + 1, projection='3d')
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2],
               c=pts[:, 2], cmap='viridis', s=1.5, alpha=0.7)
    ax.set_title(cls_name, fontsize=10, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.set_xlabel('X', fontsize=7); ax.set_ylabel('Y', fontsize=7)

plt.suptitle('ModelNet40 — Point Cloud Visualizations (colored by Z/height)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('artifact2_pointcloud_viz.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: artifact2_pointcloud_viz.png")

In [ ]:
def get_samples_per_class(data_list, n_per_class=20):
    """Collect n_per_class samples for EVERY class in the dataset"""
    class_samples = defaultdict(list)
    
    for data in data_list:
        cls = CLASSES[data.y.item()]
        if len(class_samples[cls]) < n_per_class:
            class_samples[cls].append(data)
    
    # Report what we got
    empty = [c for c in CLASSES if c not in class_samples]
    print(f"Collected samples for {len(class_samples)}/40 classes")
    if empty:
        print(f"Missing classes: {empty}")
    
    return class_samples

# Rebuild with fixed function
class_samples = get_samples_per_class(train_list, n_per_class=4)

# Now try cone again
COMPARE_CLASS = 'cone'
print(f"\nSamples for '{COMPARE_CLASS}': {len(class_samples[COMPARE_CLASS])}")

In [ ]:
COMPARE_CLASS = 'cone'

samples_to_compare = class_samples[COMPARE_CLASS][:4]  # max 4 to keep it readable
n = len(samples_to_compare)
print(f"Plotting {n} samples for '{COMPARE_CLASS}'")

fig = plt.figure(figsize=(6 * n, 5))
for i, data in enumerate(samples_to_compare):
    pts = data.pos.numpy()
    
    ax = fig.add_subplot(1, n, i+1, projection='3d')  # ← dynamic n instead of hardcoded 2
    sc = ax.scatter(pts[:,0], pts[:,1], pts[:,2],
                    c=pts[:,2], cmap='plasma', s=3, alpha=0.8, depthshade=True)
    ax.set_title(f'{COMPARE_CLASS} — Sample {i+1}', fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.set_xlim([-1,1]); ax.set_ylim([-1,1]); ax.set_zlim([-1,1])

plt.suptitle(f'Intra-class variance for: {COMPARE_CLASS}', fontsize=12)
plt.tight_layout()
plt.savefig(f'artifact2b_intraclass_{COMPARE_CLASS}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

NORMAL_CLASSES = ['table', 'chair', 'plant', 'airplane', 'cone', 'lamp']

# Collect normals for each class (first 5 samples each)
class_normals  = defaultdict(list)
counts_needed  = {c: 5 for c in NORMAL_CLASSES}

for data in train_list:                              # ← train_list
    cls = CLASSES[data.y.item()]                     # ← CLASSES
    if cls in counts_needed and counts_needed[cls] > 0:
        class_normals[cls].append(data.norm.numpy())
        counts_needed[cls] -= 1
    if all(v == 0 for v in counts_needed.values()):
        break

# Sanity check
for cls in NORMAL_CLASSES:
    print(f"{cls}: {len(class_normals[cls])} samples collected")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, cls_name in enumerate(NORMAL_CLASSES):
    if len(class_normals[cls_name]) == 0:
        print(f"WARNING: no normals for {cls_name}")
        continue

    normals = np.concatenate(class_normals[cls_name], axis=0)  # [N*1024, 3]
    ax      = axes[idx]

    h = ax.hist2d(normals[:,0], normals[:,2],
                  bins=60, cmap='hot',
                  range=[[-1,1],[-1,1]])
    fig.colorbar(h[3], ax=ax, shrink=0.8)
    ax.set_title(f'{cls_name}', fontsize=11, fontweight='bold')
    ax.set_ylabel('nz (vertical)')

    # Normal entropy — diversity score
    hist, _ = np.histogramdd(normals[:, :2], bins=20, range=[[-1,1]]*2)
    hist     = hist / hist.sum() + 1e-10
    entropy  = -np.sum(hist * np.log(hist))
    ax.set_xlabel(f'nx  |  Normal entropy: {entropy:.2f}', fontsize=9)

plt.suptitle('Normal Vector Distribution (nx vs nz) — hot = high density\n'
             'Flat objects cluster at center; complex objects spread outward',
             fontsize=12)
plt.tight_layout()
plt.savefig('artifact4_normal_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
!nvidia-smi
import torch
print(torch.cuda.get_device_name(0))
print(f"VRAM total : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"VRAM free  : {torch.cuda.memory_reserved(0) / 1e9:.2f} GB used so far")

In [ ]:
from collections import defaultdict
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

DENSITY_CLASSES = ['chair', 'airplane', 'table', 'lamp']
BINS = 50
MAX_DENSITY_SAMPLES = 50

class_all_pts        = defaultdict(list)
counts_for_density   = defaultdict(int)

for data in tqdm(train_list, desc='Aggregating points'):
    cls = CLASSES[data.y.item()]
    if cls not in DENSITY_CLASSES:
        continue
    if counts_for_density[cls] >= MAX_DENSITY_SAMPLES:
        continue
    class_all_pts[cls].append(data.pos.cpu().numpy())
    counts_for_density[cls] += 1

PROJECTIONS = [('XY (top)', 0, 1), ('XZ (front)', 0, 2), ('YZ (side)', 1, 2)]
fig, axes   = plt.subplots(len(DENSITY_CLASSES), 3,
                            figsize=(15, 4 * len(DENSITY_CLASSES)))

for row, cls_name in enumerate(DENSITY_CLASSES):
    all_pts = np.concatenate(class_all_pts[cls_name], axis=0)
    print(f"{cls_name}: {all_pts.shape[0]} total points")

    for col, (proj_name, xi, yi) in enumerate(PROJECTIONS):
        ax = axes[row][col]
        h  = ax.hist2d(all_pts[:, xi], all_pts[:, yi],
                       bins=BINS, cmap='inferno',
                       range=[[-1,1],[-1,1]])
        fig.colorbar(h[3], ax=ax, shrink=0.8)
        ax.set_title(f'{cls_name} — {proj_name}', fontsize=9, fontweight='bold')
        ax.set_xlabel(['X','X','Y'][col], fontsize=8)
        ax.set_ylabel(['Y','Z','Z'][col], fontsize=8)
        ax.set_aspect('equal')

plt.suptitle('Point Density Heatmaps (50 samples per class)\n'
             'Bright = high density = candidate spreading activation seed regions',
             fontsize=12)
plt.tight_layout()
plt.savefig('artifact6_density_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: artifact6_density_heatmaps.png")

In [ ]:
from scipy.spatial import cKDTree
from collections import defaultdict
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def chamfer_distance(pc1, pc2):
    tree1  = cKDTree(pc1)
    tree2  = cKDTree(pc2)
    d1, _  = tree2.query(pc1, k=1)
    d2, _  = tree1.query(pc2, k=1)
    return float(np.mean(d1**2) + np.mean(d2**2))

# ── All 40 classes ────────────────────────────────────────────────────
CHAMFER_CLASSES = CLASSES   # all 40, already sorted

# Get one representative point cloud per class
accum_pts      = defaultdict(list)
counts_chamfer = defaultdict(int)

for data in tqdm(train_list, desc='Collecting representatives'):
    cls = CLASSES[data.y.item()]
    if counts_chamfer[cls] >= 15:
        continue
    accum_pts[cls].append(data.pos.cpu().numpy())
    counts_chamfer[cls] += 1
    if len(counts_chamfer) == 40 and all(v >= 1 for v in counts_chamfer.values()):
        break

def get_class_representative(pts_list):
    all_pts = np.concatenate(pts_list, axis=0)          # stack all 10 samples → [10240, 3]
    idx     = np.random.choice(len(all_pts), 1024, replace=False)  # subsample back to 1024
    return all_pts[idx]

class_rep_pts = {
    cls: get_class_representative(accum_pts[cls])
    for cls in CHAMFER_CLASSES if accum_pts[cls]
}

valid_classes = [c for c in CHAMFER_CLASSES if c in class_rep_pts]
N             = len(valid_classes)
print(f"Computing {N}x{N} = {N*N} Chamfer distances...")

# ── Compute matrix ────────────────────────────────────────────────────
chamfer_matrix = np.zeros((N, N))

for i, ci in enumerate(tqdm(valid_classes, desc='Chamfer rows')):
    for j, cj in enumerate(valid_classes):
        if i == j:
            chamfer_matrix[i, j] = 0.0
        elif j > i:
            d = chamfer_distance(class_rep_pts[ci], class_rep_pts[cj])
            chamfer_matrix[i, j] = d
            chamfer_matrix[j, i] = d

# ── Plot ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(22, 18))   # larger for 40x40
mask    = np.eye(N, dtype=bool)

sns.heatmap(
    chamfer_matrix,
    xticklabels=valid_classes,
    yticklabels=valid_classes,
    cmap='YlOrRd_r',
    annot=True, fmt='.2f',        # 2 decimal places to fit in cells
    linewidths=0.3,
    annot_kws={'size': 6},        # smaller font for 40x40
    ax=ax,
    mask=mask
)
ax.set_title('Inter-class Chamfer Distance Matrix — All 40 ModelNet40 Classes\n'
             'Dark = geometrically SIMILAR = high confusion risk',
             fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0,  fontsize=8)
plt.tight_layout()
plt.savefig('artifact7_chamfer_40x40.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: artifact7_chamfer_40x40.png")

# ── Top 10 most confusable pairs ──────────────────────────────────────
print("\nTop-10 Most Geometrically Similar (Confusable) Class Pairs:")
print(f"{'Rank':<6} {'Class A':<15} {'Class B':<15} {'Chamfer Dist':>12}")
print('-' * 52)
pairs = []
for i in range(N):
    for j in range(i+1, N):
        pairs.append((chamfer_matrix[i,j], valid_classes[i], valid_classes[j]))
pairs.sort()
for rank, (d, ci, cj) in enumerate(pairs[:10], 1):
    print(f"{rank:<6} {ci:<15} {cj:<15} {d:>12.4f}")

# ── Top 10 most DISSIMILAR pairs (easiest to separate) ────────────────
print("\nTop-10 Most Geometrically Dissimilar (Easiest) Class Pairs:")
print(f"{'Rank':<6} {'Class A':<15} {'Class B':<15} {'Chamfer Dist':>12}")
print('-' * 52)
for rank, (d, ci, cj) in enumerate(reversed(pairs[-10:]), 1):
    print(f"{rank:<6} {ci:<15} {cj:<15} {d:>12.4f}")

In [ ]:
# ── SAVE FINAL CACHE with edge_index precomputed ──────────────────────
from scipy.spatial import cKDTree
import numpy as np

def add_knn_graph(data_list, k=20):
    for data in tqdm(data_list, desc='Adding kNN graphs'):
        pts  = data.pos.cpu().numpy()
        tree = cKDTree(pts)
        _, idx     = tree.query(pts, k=k+1)
        idx        = idx[:, 1:]
        N          = pts.shape[0]
        src        = np.repeat(np.arange(N), k)
        dst        = idx.flatten()
        data.edge_index = torch.tensor(
            np.stack([src, dst]), dtype=torch.long
        )
    return data_list

train_list = add_knn_graph(train_list, k=20)
test_list  = add_knn_graph(test_list,  k=20)

torch.save({
    'train'  : train_list,
    'test'   : test_list,
    'classes': CLASSES
}, '/kaggle/working/modelnet40_final.pt')

size = os.path.getsize('/kaggle/working/modelnet40_final.pt') / 1e6
print(f"Final cache saved: {size:.0f} MB")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import DataLoader
from tqdm.auto import tqdm, trange
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import numpy as np
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix
)

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE  = 32
EPOCHS      = 50
LR          = 0.001
NUM_CLASSES = 40

# ── Model ──────────────────────────────────────────────────────────────
class PointGCN(nn.Module):
    def __init__(self, in_channels=6, hidden=64, num_classes=40):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden)
        self.conv2 = GCNConv(hidden, hidden * 2)
        self.conv3 = GCNConv(hidden * 2, hidden * 4)
        self.classifier = nn.Sequential(
            nn.Linear(hidden * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.2, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=0.2, training=self.training)
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.classifier(x)

# ── DataLoader ─────────────────────────────────────────────────────────
train_loader = DataLoader(train_list, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_list,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model        = PointGCN(in_channels=6, hidden=64, num_classes=NUM_CLASSES).to(DEVICE)
optimizer    = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler    = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
total_params = sum(p.numel() for p in model.parameters())


print(f"  Parameters : {total_params:,}")
print(f"  Device     : {DEVICE}")
print(f"  Epochs     : {EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {LR}")
print(f"  Train batches : {len(train_loader)}  |  Test batches: {len(test_loader)}")
# ── Train / Test ───────────────────────────────────────────────────────
def train_epoch(model, loader, epoch_pbar):
    model.train()
    total_loss, correct, total = 0, 0, 0

    batch_pbar = tqdm(
        loader,
        desc=f'  [Train]',
        leave=False,
        ncols=90,
        bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining} {postfix}]'
    )

    for batch in batch_pbar:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        out  = model(batch)
        loss = F.cross_entropy(out, batch.y.squeeze())
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.num_graphs
        correct    += out.argmax(dim=1).eq(batch.y.squeeze()).sum().item()
        total      += batch.num_graphs

        # Live loss + acc in the batch bar
        batch_pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc' : f'{correct/total*100:.1f}%'
        })

    batch_pbar.close()
    return total_loss / total, correct / total


@torch.no_grad()
def test_epoch(model, loader):
    model.eval()
    all_preds, all_labels = [], []

    batch_pbar = tqdm(
        loader,
        desc=f'  [Test] ',
        leave=False,
        ncols=90,
        bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'
    )

    for batch in batch_pbar:
        batch  = batch.to(DEVICE)
        out    = model(batch)
        preds  = out.argmax(dim=1).cpu().numpy()
        labels = batch.y.squeeze().cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

    batch_pbar.close()

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    return {
        'OA'       : accuracy_score(all_labels, all_preds) * 100,
        'mAcc'     : balanced_accuracy_score(all_labels, all_preds) * 100,
        'macro_f1' : f1_score(all_labels, all_preds, average='macro',    zero_division=0) * 100,
        'macro_p'  : precision_score(all_labels, all_preds, average='macro', zero_division=0) * 100,
        'macro_r'  : recall_score(all_labels, all_preds, average='macro', zero_division=0) * 100,
        'w_f1'     : f1_score(all_labels, all_preds, average='weighted', zero_division=0) * 100,
        '_preds'   : all_preds,
        '_labels'  : all_labels,
    }

best_acc  = 0
best_macc = 0
history   = {k: [] for k in ['train_loss','train_acc','OA','mAcc',
                               'macro_f1','macro_p','macro_r','w_f1','lr']}

epoch_pbar = trange(
    1, EPOCHS + 1,
    desc='  Epochs',
    ncols=110,
    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining} {postfix}]'
)

for epoch in epoch_pbar:
    train_loss, train_acc = train_epoch(model, train_loader, epoch_pbar)
    test_metrics          = test_epoch(model, test_loader)
    scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']

    # Store history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc * 100)
    history['lr'].append(current_lr)
    for k in ['OA','mAcc','macro_f1','macro_p','macro_r','w_f1']:
        history[k].append(test_metrics[k])

    # Track best
    if test_metrics['OA'] > best_acc:
        best_acc  = test_metrics['OA']
        best_macc = test_metrics['mAcc']
        best_metrics = test_metrics
        torch.save(model.state_dict(), '/kaggle/working/best_gcn.pt')

    # Update outer epoch bar with all key metrics
    epoch_pbar.set_postfix({
        'loss'    : f"{train_loss:.4f}",
        'trAcc'   : f"{train_acc*100:.1f}%",
        'OA'      : f"{test_metrics['OA']:.1f}%",
        'mAcc'    : f"{test_metrics['mAcc']:.1f}%",
        'F1'      : f"{test_metrics['macro_f1']:.1f}%",
        'best'    : f"{best_acc:.1f}%",
    })

epoch_pbar.close()

print("Final results")
print(f"  Overall Accuracy (OA)   : {best_acc:.2f}%   ← micro")
print(f"  Mean Class Acc  (mAcc)  : {best_macc:.2f}%   ← macro")
print(f"  Macro Precision         : {best_metrics['macro_p']:.2f}%")
print(f"  Macro Recall            : {best_metrics['macro_r']:.2f}%")
print(f"  Macro F1                : {best_metrics['macro_f1']:.2f}%")
print(f"  Weighted F1             : {best_metrics['w_f1']:.2f}%")
print(f"  DGCNN   → OA: 92.9%  mAcc: 90.2%")
print(f"  PointNet → OA: 89.2%  mAcc: 86.2%")
print(f"  Gap (OA)  : {0.929 - best_acc/100:.4f} ({(0.929 - best_acc/100)*100:.2f}%)")


In [ ]:

BG      = '#0F1117'
CARD    = '#1A1D27'
A1      = '#4F8EF7'   # blue
A2      = '#F76F8E'   # pink
A3      = '#FFD166'   # gold
A4      = '#00C896'   # green
A5      = '#B48EF7'   # purple
GRID    = '#2A2D3A'
TEXT    = '#E0E4F0'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': CARD,
    'axes.edgecolor'  : GRID,'axes.labelcolor': TEXT,
    'xtick.color'     : TEXT,'ytick.color'    : TEXT,
    'text.color'      : TEXT,'grid.color'     : GRID,
    'grid.linewidth'  : 0.5, 'font.family'    : 'monospace',
})

epochs_x = np.arange(1, EPOCHS + 1)
fig      = plt.figure(figsize=(26, 22), facecolor=BG)
fig.suptitle('PointGCN — Full Training & Evaluation Dashboard\n'
             'ModelNet40 · 40 Classes · 1024 pts · k=20 · 13.4× Imbalance',
             fontsize=14, fontweight='bold', color=TEXT, y=0.99)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.42, wspace=0.35)

# ── 1. OA + mAcc curves ────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(epochs_x, history['OA'],   color=A2, lw=2.5, label='Test OA  (micro)')
ax1.plot(epochs_x, history['mAcc'], color=A3, lw=2.5, label='Test mAcc (macro)')
ax1.plot(epochs_x, history['train_acc'], color=A1, lw=1.5,
         linestyle='--', alpha=0.7, label='Train Acc')
ax1.fill_between(epochs_x, history['OA'], history['mAcc'],
                 alpha=0.08, color=A3,
                 label=f'OA−mAcc gap (imbalance effect)')
for name, val, col in [('DGCNN OA',92.9,A4),('DGCNN mAcc',90.2,'#88FFCC'),
                        ('PointNet OA',89.2,'#FFA500')]:
    ax1.axhline(val, color=col, linestyle=':', lw=1.2, alpha=0.8, label=f'{name} {val}%')
best_ep = np.argmax(history['OA'])
ax1.scatter([best_ep+1],[history['OA'][best_ep]], color='white', s=90, zorder=5)
ax1.annotate(f"Best OA\n{history['OA'][best_ep]:.1f}%",
             xy=(best_ep+1, history['OA'][best_ep]),
             xytext=(best_ep+3, history['OA'][best_ep]-8),
             color='white', fontsize=7.5,
             arrowprops=dict(arrowstyle='->', color='white', lw=0.8))
ax1.set_title('Overall Accuracy (OA) vs Mean Class Accuracy (mAcc)', fontsize=11, pad=8)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy (%)')
ax1.set_ylim([0, 100]); ax1.grid(True, alpha=0.35)
ax1.legend(fontsize=7.5, loc='lower right', facecolor=CARD, edgecolor=GRID, ncol=2)

# ── 2. F1 / Precision / Recall ─────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, :2])
ax2.plot(epochs_x, history['macro_f1'], color=A4,  lw=2.5, label='Macro F1')
ax2.plot(epochs_x, history['macro_p'],  color=A5,  lw=2,   label='Macro Precision')
ax2.plot(epochs_x, history['macro_r'],  color=A3,  lw=2,   label='Macro Recall')
ax2.plot(epochs_x, history['w_f1'],     color=A1,  lw=1.5,
         linestyle='--', alpha=0.7, label='Weighted F1')
ax2.set_title('Macro Precision / Recall / F1  (each class weighted equally)', fontsize=11, pad=8)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Score (%)')
ax2.set_ylim([0, 100]); ax2.grid(True, alpha=0.35)
ax2.legend(fontsize=8, facecolor=CARD, edgecolor=GRID)

# ── 3. Loss + LR ───────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[2, :2])
ax3_twin = ax3.twinx()
ax3.plot(epochs_x, history['train_loss'], color=A3, lw=2, label='Train Loss')
ax3.fill_between(epochs_x, history['train_loss'], alpha=0.12, color=A3)
ax3_twin.step(epochs_x, history['lr'], color=A5, lw=1.5,
              where='post', linestyle='--', alpha=0.8, label='LR')
ax3.set_title('Training Loss + Learning Rate Schedule', fontsize=11, pad=8)
ax3.set_xlabel('Epoch'); ax3.set_ylabel('Loss', color=A3)
ax3_twin.set_ylabel('Learning Rate', color=A5)
ax3.grid(True, alpha=0.35)
lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax3_twin.get_legend_handles_labels()
ax3.legend(lines1+lines2, labels1+labels2, fontsize=8,
           facecolor=CARD, edgecolor=GRID)

# ── 4. Benchmark bar chart ─────────────────────────────────────────────
ax4 = fig.add_subplot(gs[0, 2])
model_names = ['Random', 'Our GCN\n(OA)', 'Our GCN\n(mAcc)',
               'PointNet\n(OA)', 'PointNet++\n(OA)', 'DGCNN\n(OA)']
accs        = [2.5, best_acc, best_macc, 89.2, 91.9, 92.9]
colors      = ['#445566', A2, A3, '#FFA500', '#FF6B6B', A4]
bars        = ax4.barh(model_names, accs, color=colors,
                       height=0.55, edgecolor=BG, linewidth=1.5)
for bar, acc in zip(bars, accs):
    ax4.text(min(bar.get_width()+0.8, 97),
             bar.get_y()+bar.get_height()/2,
             f'{acc:.1f}%', va='center', fontsize=8, color=TEXT, fontweight='bold')
ax4.set_xlim([0, 100])
ax4.set_title('Benchmark Comparison', fontsize=11, pad=8)
ax4.set_xlabel('Accuracy (%)')
ax4.axvline(best_acc, color=A2, linestyle='--', alpha=0.4)
ax4.grid(True, alpha=0.3, axis='x')

# ── 5. Per-class accuracy bar ──────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
per_class = {}
preds_arr  = best_metrics['_preds']
labels_arr = best_metrics['_labels']
for i, cls in enumerate(CLASSES):
    mask = labels_arr == i
    per_class[cls] = (preds_arr[mask] == i).mean() * 100 if mask.sum() > 0 else 0.0

sorted_cls  = sorted(per_class.items(), key=lambda x: x[1])
cls_names   = [x[0] for x in sorted_cls]
cls_accs    = [x[1] for x in sorted_cls]
bar_cols    = [A4 if a >= 80 else (A3 if a >= 50 else A2) for a in cls_accs]
ax5.barh(cls_names, cls_accs, color=bar_cols, height=0.75, edgecolor=BG)
ax5.axvline(np.mean(cls_accs), color='white', linestyle='--', lw=1,
            alpha=0.6, label=f'Mean {np.mean(cls_accs):.1f}%')
ax5.set_title('Per-Class Accuracy\n(green≥80%, yellow≥50%, red<50%)', fontsize=9, pad=8)
ax5.set_xlabel('Accuracy (%)')
ax5.set_xlim([0, 110])
ax5.tick_params(axis='y', labelsize=6.5)
ax5.legend(fontsize=7, facecolor=CARD, edgecolor=GRID)

# ── 6. Summary stats card ──────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 2])
ax6.axis('off')
stats = [
    ('Best OA  (micro)',  f"{best_acc:.2f}%"),
    ('Best mAcc (macro)', f"{best_macc:.2f}%"),
    ('OA − mAcc gap',     f"{best_acc-best_macc:.2f}%  ← imbalance"),
    ('Macro F1',          f"{best_metrics['macro_f1']:.2f}%"),
    ('Macro Precision',   f"{best_metrics['macro_p']:.2f}%"),
    ('Macro Recall',      f"{best_metrics['macro_r']:.2f}%"),
    ('Weighted F1',       f"{best_metrics['w_f1']:.2f}%"),
    ('Gap to DGCNN (OA)', f"{92.9-best_acc:.2f}%"),
    ('Best class',        max(per_class, key=per_class.get)),
    ('Worst class',       min(per_class, key=per_class.get)),
    ('Parameters',        f'{total_params:,}'),
    ('Epochs',            f'{EPOCHS}'),
]
ax6.text(0.5, 1.03, 'Run Summary', ha='center', fontsize=11,
         fontweight='bold', color=TEXT, transform=ax6.transAxes)
y_pos = 0.94
for i, (k, v) in enumerate(stats):
    bg = '#1E2235' if i % 2 == 0 else CARD
    ax6.add_patch(plt.Rectangle((0, y_pos-0.075), 1, 0.075,
                                  transform=ax6.transAxes, color=bg, zorder=0))
    ax6.text(0.04, y_pos-0.038, k, ha='left', va='center',
             fontsize=7.5, color='#8899BB', transform=ax6.transAxes)
    val_color = A4 if any(c.isdigit() and float(''.join(
        filter(lambda x: x in '0123456789.', v or '0'))) > 85
        for c in [v[0]] if c.isdigit()) else TEXT
    ax6.text(0.96, y_pos-0.038, str(v), ha='right', va='center',
             fontsize=7.5, fontweight='bold', color=TEXT, transform=ax6.transAxes)
    y_pos -= 0.075

plt.savefig('gcn_full_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor=BG, edgecolor='none')
plt.show()
print("Saved: gcn_full_dashboard.png")
